In [1]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
import time, os, json
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from thop import profile as thop_profile

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)
SAVE_DIR   = ""
OUTPUT_JSON = "all_models_eval.json"

# ── Architecture ───────────────────────────────────────────────
def build_model(num_classes=10):
    model = models.resnet50(weights=None)
    model.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc      = nn.Linear(model.fc.in_features, num_classes)
    return model


# ── Architecture ───────────────────────────────────────────────
def build_resnet50(num_classes=10):
    model = models.resnet50(weights=None)
    model.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc      = nn.Linear(model.fc.in_features, num_classes)
    return model

def build_resnet18(num_classes=10):
    model = models.resnet18(weights=None)
    model.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc      = nn.Linear(model.fc.in_features, num_classes)
    return model



# ── Data ───────────────────────────────────────────────────────
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])
test_set    = torchvision.datasets.CIFAR10(root='../../data', train=False,
                                            download=True, transform=transform_test)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=128,
                                           shuffle=False, num_workers=0)

# ── Models to evaluate ─────────────────────────────────────────
MODELS = {
    "1_student_resnet18_attention_layer3_beta100"  : ("student_resnet18_attention_layer3_beta100.pth", "resnet18"),
    "2_student_resnet18_fitnet_layer3_beta100" : ("student_resnet18_fitnet_layer3_beta100.pth", "resnet18"),
    "3_student_resnet18_nst_layer3_beta100": ("student_resnet18_nst_layer3_beta100.pth", "resnet18"),
}

all_results = {}

for key, (fname, arch) in MODELS.items():
    SAVE_PATH = os.path.join(SAVE_DIR, fname)

    if not os.path.exists(SAVE_PATH):
        print(f"⚠  SKIP {key} — not found: {SAVE_PATH}")
        continue

    print(f"\n{'='*55}")
    print(f"  {key}")
    print(f"{'='*55}")

    # ── Load correct architecture ──────────────────────────────
    model = build_resnet18() if arch == "resnet18" else build_resnet50()
    model.load_state_dict(torch.load(SAVE_PATH, map_location=DEVICE))
    model = model.to(DEVICE).eval()

    # ── Classification metrics ─────────────────────────────────
    all_preds, all_labels = [], []
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(DEVICE)
            all_preds.extend(model(inputs).argmax(1).cpu().numpy())
            all_labels.extend(labels.numpy())

    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)

    acc       = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average='macro')
    recall    = recall_score(all_labels, all_preds, average='macro')
    f1        = f1_score(all_labels, all_preds, average='macro')

    # ── FLOPs ──────────────────────────────────────────────────
    dummy_flops = torch.randn(1, 3, 32, 32).to(DEVICE)
    macs, _     = thop_profile(model, inputs=(dummy_flops,), verbose=False)
    flops_G     = macs * 2 / 1e9

    # ── Parameters & disk size ─────────────────────────────────
    total_params = sum(p.numel() for p in model.parameters())
    nonzero_params = int(sum((p != 0).sum().item() for p in model.parameters()))
    size_mb      = os.path.getsize(SAVE_PATH) / 1e6

    # ── Single image inference ─────────────────────────────────
    dummy_single = torch.randn(1, 3, 32, 32, device=DEVICE)
    with torch.no_grad():
        for _ in range(50):
            model(dummy_single)
    torch.cuda.synchronize()

    times = []
    with torch.no_grad():
        for _ in range(500):
            torch.cuda.synchronize()
            t0 = time.perf_counter()
            model(dummy_single)
            torch.cuda.synchronize()
            times.append(time.perf_counter() - t0)
    inf_ms_single = np.mean(times) * 1000

    # ── Batch 128 inference ────────────────────────────────────
    dummy_batch = torch.randn(128, 3, 32, 32, device=DEVICE)
    start_ev    = torch.cuda.Event(enable_timing=True)
    end_ev      = torch.cuda.Event(enable_timing=True)

    with torch.no_grad():
        for _ in range(10):
            model(dummy_batch)
    torch.cuda.synchronize()

    batch_times = []
    with torch.no_grad():
        for _ in range(100):
            start_ev.record()
            model(dummy_batch)
            end_ev.record()
            torch.cuda.synchronize()
            batch_times.append(start_ev.elapsed_time(end_ev))

    inf_ms_batch128     = np.mean(batch_times)
    inf_ms_per_img      = inf_ms_batch128 / 128
    throughput_imgs_sec = 128 / (inf_ms_batch128 / 1000)

    # ── Print ──────────────────────────────────────────────────
    print(f"  Accuracy    : {acc:.4f}")
    print(f"  Precision   : {precision:.4f}")
    print(f"  Recall      : {recall:.4f}")
    print(f"  F1          : {f1:.4f}")
    print(f"  Params      : {total_params:,}  (nonzero: {nonzero_params:,})")
    print(f"  Sparsity    : {1 - nonzero_params/total_params:.3f}")
    print(f"  Size        : {size_mb:.2f} MB")
    print(f"  FLOPs       : {flops_G:.3f} GFLOPs")
    print(f"  Single img  : {inf_ms_single:.3f} ms")
    print(f"  Batch-128   : {inf_ms_batch128:.2f} ms")
    print(f"  Per-image   : {inf_ms_per_img:.3f} ms")
    print(f"  Throughput  : {throughput_imgs_sec:.1f} imgs/sec")

    # ── Store ──────────────────────────────────────────────────
    all_results[key] = {
        "accuracy"   : acc,
        "precision"  : precision,
        "recall"     : recall,
        "f1"         : f1,
        "params"     : total_params,
        "params_nonzero": nonzero_params,
        "sparsity_actual": float(1 - nonzero_params / total_params),
        "size_mb"    : size_mb,
        "flops_G"    : flops_G,
        "inference_ms": {
            "single_img_gpu_ms"  : round(inf_ms_single, 4),
            "batch128_gpu_ms"    : round(inf_ms_batch128, 4),
            "per_img_gpu_ms"     : round(inf_ms_per_img, 4),
            "throughput_imgs_sec": round(throughput_imgs_sec, 1),
            "timing_method"      : "CUDA events + torch.cuda.synchronize()",
        },
    }

# ── Save all to one JSON ───────────────────────────────────────
with open(OUTPUT_JSON, "w") as f:
    json.dump(all_results, f, indent=2)

print(f"\n✓ All results saved → {OUTPUT_JSON}")

# ── Summary table ──────────────────────────────────────────────
print(f"\n{'Method':<28} {'Acc':>7} {'F1':>7} {'Spar':>6} {'MB':>7} {'GFLOPs':>8} {'Single ms':>10} {'Imgs/s':>8}")
print("-" * 88)
for k, m in all_results.items():
    inf = m['inference_ms']
    print(f"{k:<28} {m['accuracy']:>7.4f} {m['f1']:>7.4f} "
          f"{m['sparsity_actual']:>6.3f} {m['size_mb']:>7.2f} "
          f"{m['flops_G']:>8.3f} {inf['single_img_gpu_ms']:>10.3f} "
          f"{inf['throughput_imgs_sec']:>8.1f}")

e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")



  1_student_resnet18_attention_layer3_beta100
  Accuracy    : 0.8952
  Precision   : 0.8951
  Recall      : 0.8952
  F1          : 0.8949
  Params      : 11,173,962  (nonzero: 11,173,962)
  Sparsity    : 0.000
  Size        : 44.78 MB
  FLOPs       : 1.116 GFLOPs
  Single img  : 4.970 ms
  Batch-128   : 22.63 ms
  Per-image   : 0.177 ms
  Throughput  : 5656.0 imgs/sec

  2_student_resnet18_fitnet_layer3_beta100
  Accuracy    : 0.7560
  Precision   : 0.7533
  Recall      : 0.7560
  F1          : 0.7535
  Params      : 11,173,962  (nonzero: 11,173,962)
  Sparsity    : 0.000
  Size        : 44.78 MB
  FLOPs       : 1.116 GFLOPs
  Single img  : 4.885 ms
  Batch-128   : 22.29 ms
  Per-image   : 0.174 ms
  Throughput  : 5742.3 imgs/sec

  3_student_resnet18_nst_layer3_beta100
  Accuracy    : 0.8859
  Precision   : 0.8859
  Recall      : 0.8859
  F1          : 0.8857
  Params      : 11,173,962  (nonzero: 11,173,962)
  Sparsity    : 0.000
  Size        : 44.78 MB
  FLOPs       : 1.116 GFLOPs
